# Nova AI — Qwen ko "Nova" banana (free GPU par)

Yeh notebook **Qwen2.5-1.5B** ko Nova ke data par train karta hai aur trained model aapke **Hugging Face** account par (private) upload kar deta hai. Wahan se RunPod use chala sakta hai.

### Ek baar ki taiyaari
1. **Hugging Face** (huggingface.co) par account banaiye → *Settings → Access Tokens → Create new token* → type **Write** → token copy kariye.
2. **Kaggle** (kaggle.com) par account banaiye aur **phone number verify** kariye (bina iske GPU aur internet nahi milta).
3. Is notebook ko Kaggle par kholiye (*Create → New Notebook → File → Import Notebook*).
4. Daayein panel me:
   - **Accelerator → GPU T4** chuniye
   - **Internet → On**
5. *Add-ons → Secrets* me naya secret: naam **`HF_TOKEN`**, value = aapka token.
6. Neeche `HF_REPO` me apna Hugging Face username likhiye.
7. **Run All** dabaiye. Lagbhag 15–30 minute lagenge.

> Colab par bhi chalta hai: *Runtime → T4 GPU*, aur token Colab ke 🔑 *Secrets* me `HF_TOKEN` naam se daaliye.


In [ ]:
# Apna Hugging Face username yahan likhiye 👇
HF_REPO = "AAPKA-HF-USERNAME/nova-1.5b"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EPOCHS = 3

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

# HF_TOKEN: Kaggle ya Colab ke secrets se, warna haath se
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

assert "AAPKA-HF-USERNAME" not in HF_REPO, "Upar HF_REPO me apna Hugging Face username likhiye"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # ek GPU kaafi hai, aur training seedhi rehti hai
print("Token mil gaya ✓")

In [ ]:
!git clone --depth 1 https://github.com/mangalkumar71643-creator/Nova-ai-.git nova
%cd nova/model
!pip install -q "transformers>=5.0" "peft>=0.13" "accelerate>=1.0" "huggingface_hub>=0.25"
!wc -l data/*.jsonl

## Training
Model sirf Nova ke jawabon se seekhta hai. Khatam hote hi model `out/nova-lora-merged` me banega aur Hugging Face par upload hoga.

In [ ]:
!python train.py --base {BASE_MODEL} --epochs {EPOCHS} --batch 4 --grad-accum 2 --max-len 1024 --push-to-hub {HF_REPO}

## Test: Nova se baat kariye

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

path = "out/nova-lora-merged"
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16, device_map="cuda")

SYSTEM = "You are Nova AI, a helpful, friendly assistant. Reply in the same language the user writes in (Hindi, Hinglish or English)."

def nova(question):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for q in ["Tum kaun ho?", "Tumhe kisne banaya?", "Photo kaise bhejun?", "Photosynthesis kya hota hai?"]:
    print("🙋", q)
    print("🤖", nova(q), "\n")

## Aage kya
1. Hugging Face par model dikh raha hai: `https://huggingface.co/<HF_REPO>` (private).
2. **RunPod → Serverless → New Endpoint → vLLM**:
   - Model: aapka `HF_REPO` (jaise `aapka-naam/nova-1.5b`)
   - Model private hai, isliye Environment Variables me **`HF_TOKEN`** daaliye (yahi token)
   - **Active workers = 0**, Max workers = 1
3. Endpoint ID Nova ke developer ko bhejiye. Vercel server ko jod diya jaayega aur app aapke **apne Nova model** se jawab dega. 🎉
